# Анализ ошибок и статистическая неопределённость

Интерактивная замена `error_analysis.py`. Используются только сохранённые prediction CSV из `reports/final_evaluation/`; модели не обучаются, а bootstrap не пересчитывается.


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

REPORTS = Path('reports') if Path('reports').exists() else Path('../reports')
ROOT = REPORTS / 'final_evaluation'; OUT = REPORTS / 'error_analysis'
keys = ['logreg', 'mlp_original', 'mlp_regularized', 'lstm']
names = {'logreg':'Logistic Regression', 'mlp_original':'Original MLP', 'mlp_regularized':'Regularized MLP', 'lstm':'LSTM'}
predictions = {key: pd.read_csv(ROOT / f'{key}_predictions.csv').sort_values('id').reset_index(drop=True) for key in keys}
reference = predictions['logreg']
assert all(np.array_equal(reference.id, frame.id) and np.array_equal(reference.true_label, frame.true_label) for frame in predictions.values())


In [ ]:
rows = []
for key, frame in predictions.items():
    tn, fp, fn, tp = confusion_matrix(frame.true_label, frame.predicted_label, labels=[0, 1]).ravel()
    rows.append({'model': names[key], 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn, 'FPR': fp/(fp+tn), 'FNR': fn/(fn+tp), 'F1': f1_score(frame.true_label, frame.predicted_label), 'ROC-AUC': roc_auc_score(frame.true_label, frame.probability_ai)})
error_summary = pd.DataFrame(rows); error_summary


In [ ]:
merged = reference[['id','true_label','predicted_label']].rename(columns={'predicted_label':'logreg_pred'})
for key in keys[1:]: merged = merged.merge(predictions[key][['id','true_label','predicted_label']].rename(columns={'true_label':f'{key}_label','predicted_label':f'{key}_pred'}), on='id', validate='one_to_one')
wrong = np.logical_and.reduce([merged[f'{key}_pred'].to_numpy() != merged.true_label.to_numpy() for key in keys])
common_errors = merged.loc[wrong, ['id','true_label','logreg_pred','mlp_original_pred','mlp_regularized_pred','lstm_pred']]
print(f'Общие ошибки всех четырёх моделей: {len(common_errors)}')
common_errors.head()


In [ ]:
length = pd.read_csv(OUT / 'length_analysis.csv')
fig, ax = plt.subplots(figsize=(7, 4))
for model, group in length.groupby('model'): ax.plot(group.length_group, group.f1, marker='o', label=model)
ax.set(xlabel='Квартиль длины текста', ylabel='F1', ylim=(0,1), title='F1 в группах длины'); ax.legend(fontsize=8); ax.grid(alpha=.25)
plt.show()


In [ ]:
pairs = pd.read_csv(OUT / 'bootstrap_pairwise_differences.csv')
pairs
# Это сохранённый paired class-stratified bootstrap (2 000 повторов, seed=42); пересчёт не выполняется.
